<a href="https://colab.research.google.com/github/pmlade23-droid/Analyse-Sentiment-IMDb./blob/main/projet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets pandas scikit-learn


In [7]:
import pandas as pd

print("1. Téléchargement direct du dataset d'avis clients depuis GitHub...")

# URL d'un miroir public ultra-stable du dataset de 50 000 avis
url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"

# Chargement direct avec Pandas
df_raw = pd.read_csv(url)

print("-> Dataset chargé avec succès !")

# 2. sous-ensemble de 20 000 lignes équilibrées
# (10 000 positifs et 10 000 négatifs) pour la vectorisation TF-IDF
df_pos = df_raw[df_raw['sentiment'] == 'positive'].sample(n=10000, random_state=42)
df_neg = df_raw[df_raw['sentiment'] == 'negative'].sample(n=10000, random_state=42)

# Fusion pour avoir notre dataset final
df = pd.concat([df_pos, df_neg]).reset_index(drop=True)

# Conversion des labels textuels en nombres (0 = négatif, 1 = positif)
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print(f"-> Le jeu de données final contient {len(df)} avis de films.")

# ==========================================
# EXPLORATION RAPIDE DES DONNÉES
# ==========================================
print("\n=== EXPLORATION DES DONNÉES ===")
print(f"Volume : {df.shape[0]} lignes.")
print("\nRépartition des sentiments (0 = Négatif, 1 = Positif) :")
print(df['sentiment'].value_counts(normalize=True) * 100)

# Calcul de la longueur moyenne des avis en mots
df['word_count'] = df['review'].apply(lambda x: len(str(x).split()))
print("\nLongueur moyenne des avis (en mots) :", round(df['word_count'].mean(), 1))

print("\n--- Aperçu rapide d'un avis ---")
print(df['review'].iloc[0][:250] + "...")

1. Téléchargement direct du dataset d'avis clients depuis GitHub...
-> Dataset chargé avec succès !
-> Le jeu de données final contient 20000 avis de films.

=== EXPLORATION DES DONNÉES ===
Volume : 20000 lignes.

Répartition des sentiments (0 = Négatif, 1 = Positif) :
sentiment
1    50.0
0    50.0
Name: proportion, dtype: float64

Longueur moyenne des avis (en mots) : 231.3

--- Aperçu rapide d'un avis ---
I don't know how or why this film has a meager rating on IMDb. This film, accompanied by "I am Curious: Blue" is a masterwork.<br /><br />The only thing that will let you down in this film is if you don't like the process of film, don't like psycholo...


In [9]:
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Téléchargement de la liste des mots vides (Stopwords) en anglais
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# 2. Fonction de nettoyage de texte personnalisée
def clean_english_text(text):
    # Enlever les balises HTML (ex: <br />)
    text = re.sub(r'<[^>]+>', ' ', text)
    # Passage en minuscules
    text = text.lower()
    # Remplacement de la ponctuation et des chiffres par un espace
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Découpage en mots (Tokenisation)
    words = text.split()
    # Suppression des stopwords anglais
    cleaned_words = [word for word in words if word not in stop_words]
    # Reconstitution de la chaîne de caractères
    return " ".join(cleaned_words)

print("2. Nettoyage des 20 000 avis en cours (cela peut prendre quelques secondes)...")
df['cleaned_review'] = df['review'].apply(clean_english_text)
print("-> Nettoyage terminé !")

# Comparaison Avant / Après sur le premier avis
print("\n--- AVANT NETTOYAGE ---")
print(df['review'].iloc[0][:200] + "...")
print("\n--- APRÈS NETTOYAGE ---")
print(df['cleaned_review'].iloc[0][:200] + "...")

# ==========================================
# 3. DIVISION DU JEU DE DONNÉES (TRAIN / TEST)
# ==========================================
# On garde 80% pour l'entraînement et 20% pour tester notre futur modèle
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_review'],
    df['sentiment'],
    test_size=0.20,
    random_state=42
)

# ==========================================
# 4. VECTORISATION TF-IDF
# ==========================================
print("\n3. Vectorisation TF-IDF en cours...")
# On limite à 5000 mots les plus importants pour éviter d'avoir une matrice trop géante
tfidf = TfidfVectorizer(max_features=5000)

# On apprend le vocabulaire et on transforme le jeu d'entraînement
X_train_tfidf = tfidf.fit_transform(X_train)
# On transforme le jeu de test (sans réapprendre le vocabulaire)
X_test_tfidf = tfidf.transform(X_test)

print("-> Vectorisation terminée avec succès !")
print(f"Dimensions de la matrice d'entraînement : {X_train_tfidf.shape} (16 000 avis, 5 000 mots-clés)")
print(f"Dimensions de la matrice de test : {X_test_tfidf.shape} (4 000 avis, 5 000 mots-clés)")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


2. Nettoyage des 20 000 avis en cours (cela peut prendre quelques secondes)...
-> Nettoyage terminé !

--- AVANT NETTOYAGE ---
I don't know how or why this film has a meager rating on IMDb. This film, accompanied by "I am Curious: Blue" is a masterwork.<br /><br />The only thing that will let you down in this film is if you d...

--- APRÈS NETTOYAGE ---
know film meager rating imdb film accompanied curious blue masterwork thing let film like process film like psychology expecting hardcore pornographic ramming film want watch unwind film want see like...

3. Vectorisation TF-IDF en cours...
-> Vectorisation terminée avec succès !
Dimensions de la matrice d'entraînement : (16000, 5000) (16 000 avis, 5 000 mots-clés)
Dimensions de la matrice de test : (4000, 5000) (4 000 avis, 5 000 mots-clés)


In [10]:
# ==========================================
# BLOC DE TEST : VÉRIFICATION DU TF-IDF
# ==========================================

print("=== VÉRIFICATION DE LA VECTORISATION TF-IDF ===\n")

# 1. Récupérer le dictionnaire des mots appris (vocabulaire)
vocabulaire = tfidf.get_feature_names_out()
print(f"Nombre total de mots uniques retenus : {len(vocabulaire)}")
print(f"Exemple de quelques mots de votre vocabulaire : {list(vocabulaire[1000:1015])}\n")

# 2. Inspecter un avis spécifique après transformation
index_avis = 0  # Vous pouvez changer cet index pour tester un autre avis (de 0 à 15999)

# Récupérer l'avis original nettoyé
avis_nettoye = X_train.iloc[index_avis]
# Récupérer sa ligne vectorisée sous forme de liste de scores
scores_tfidf = X_train_tfidf[index_avis].toarray()[0]

# Associer chaque mot présent dans cet avis à son score TF-IDF
mots_et_scores = []
for idx_mot, score in enumerate(scores_tfidf):
    if score > 0:  # On ne garde que les mots présents dans cet avis
        mots_et_scores.append((vocabulaire[idx_mot], round(score, 4)))

# Trier les mots par leur importance (score TF-IDF décroissant)
mots_et_scores_tries = sorted(mots_et_scores, key=lambda x: x[1], reverse=True)

print(f"--- ANALYSE DE L'AVIS N°{index_avis} ---")
print(f"Texte nettoyé : \n\"{avis_nettoye[:200]}...\"\n")
print("Top 10 des mots les plus discriminants (importants) dans cet avis selon le TF-IDF :")
for mot, score in mots_et_scores_tries[:10]:
    print(f"  * Mots: '{mot}' ---> Score TF-IDF: {score}")

=== VÉRIFICATION DE LA VECTORISATION TF-IDF ===

Nombre total de mots uniques retenus : 5000
Exemple de quelques mots de votre vocabulaire : ['cousin', 'cover', 'covered', 'covers', 'cowboy', 'cox', 'crack', 'craft', 'crafted', 'craig', 'crap', 'crappy', 'crash', 'crawford', 'crazed']

--- ANALYSE DE L'AVIS N°0 ---
Texte nettoyé : 
"initially dubious movie merely subject richly drawn characters fabulous scenes buffalo hunt dramatic conclusion make well worth watching initially trouble distinguishing two buffalo hunters movie prog..."

Top 10 des mots les plus discriminants (importants) dans cet avis selon le TF-IDF :
  * Mots: 'initially' ---> Score TF-IDF: 0.5031
  * Mots: 'hunters' ---> Score TF-IDF: 0.2842
  * Mots: 'increasingly' ---> Score TF-IDF: 0.2636
  * Mots: 'fabulous' ---> Score TF-IDF: 0.2491
  * Mots: 'hunt' ---> Score TF-IDF: 0.2447
  * Mots: 'haunted' ---> Score TF-IDF: 0.244
  * Mots: 'merely' ---> Score TF-IDF: 0.2251
  * Mots: 'drawn' ---> Score TF-IDF: 0.2133
  * Mo

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("1. Vectorisation TF-IDF optimisée en cours (mots uniques + couples de mots)...")

# Configuration avancée du TF-IDF
tfidf = TfidfVectorizer(
    max_features=5000,      # On garde les 5000 meilleurs éléments
    min_df=5,               # Ignore les mots qui apparaissent dans moins de 5 avis
    ngram_range=(1, 2)      # Capture les mots uniques ET les paires de mots
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print("-> Vectorisation optimisée terminée avec succès !")
print(f"Dimensions de la matrice d'entraînement : {X_train_tfidf.shape}")

1. Vectorisation TF-IDF optimisée en cours (mots uniques + couples de mots)...
-> Vectorisation optimisée terminée avec succès !
Dimensions de la matrice d'entraînement : (16000, 5000)


In [13]:

print("=== VÉRIFICATION DU TF-IDF OPTIMISÉ ===\n")

vocabulaire = tfidf.get_feature_names_out()
print(f"Nombre total de termes retenus : {len(vocabulaire)}")

# On inspecte à nouveau l'avis n°0
index_avis = 0
avis_nettoye = X_train.iloc[index_avis]
scores_tfidf = X_train_tfidf[index_avis].toarray()[0]

mots_et_scores = []
for idx_mot, score in enumerate(scores_tfidf):
    if score > 0:
        mots_et_scores.append((vocabulaire[idx_mot], round(score, 4)))

mots_et_scores_tries = sorted(mots_et_scores, key=lambda x: x[1], reverse=True)

print(f"--- ANALYSE DE L'AVIS N°{index_avis} ---")
print(f"Texte nettoyé : \n\"{avis_nettoye[:250]}...\"\n")
print("Top 10 des termes les plus importants:")
for mot, score in mots_et_scores_tries[:10]:
    print(f"  * Terme: '{mot}' ---> Score TF-IDF: {score}")

=== VÉRIFICATION DU TF-IDF OPTIMISÉ ===

Nombre total de termes retenus : 5000
--- ANALYSE DE L'AVIS N°0 ---
Texte nettoyé : 
"initially dubious movie merely subject richly drawn characters fabulous scenes buffalo hunt dramatic conclusion make well worth watching initially trouble distinguishing two buffalo hunters movie progressed increasingly distinguished still haunted fi..."

Top 10 des termes les plus importants:
  * Terme: 'initially' ---> Score TF-IDF: 0.4823
  * Terme: 'final scene' ---> Score TF-IDF: 0.2549
  * Terme: 'increasingly' ---> Score TF-IDF: 0.2527
  * Terme: 'fabulous' ---> Score TF-IDF: 0.2388
  * Terme: 'hunt' ---> Score TF-IDF: 0.2345
  * Terme: 'haunted' ---> Score TF-IDF: 0.2339
  * Terme: 'well worth' ---> Score TF-IDF: 0.2242
  * Terme: 'merely' ---> Score TF-IDF: 0.2157
  * Terme: 'drawn' ---> Score TF-IDF: 0.2044
  * Terme: 'worth watching' ---> Score TF-IDF: 0.2002


In [14]:
# ==========================================
# APERÇU DÉTAILLÉ DE LA BASE DE DONNÉES
# ==========================================

print("=== APERÇU DE VOTRE BASE DE DONNÉES (DATAFRAME) ===")

# Active l'affichage interactif des tableaux dans Google Colab
try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
except ImportError:
    pass

# 1. Afficher les 5 premières lignes du dataset
print("\n--- Les 5 premières lignes du tableau ---")
display(df[['review', 'cleaned_review', 'sentiment']].head())

# 2. Afficher un exemple d'avis positif et un exemple d'avis négatif côte à côte
print("\n--- Comparaison d'un avis Positif et d'un avis Négatif ---")
positive_sample = df[df['sentiment'] == 1].iloc[0]
negative_sample = df[df['sentiment'] == 0].iloc[0]

print("\n[AVIS POSITIF ORIGINAL] :")
print(positive_sample['review'][:300] + "...")
print("\n[AVIS POSITIF NETTOYÉ] :")
print(positive_sample['cleaned_review'][:300] + "...")

print("\n" + "="*50)

print("\n[AVIS NÉGATIF ORIGINAL] :")
print(negative_sample['review'][:300] + "...")
print("\n[AVIS NÉGATIF NETTOYÉ] :")
print(negative_sample['cleaned_review'][:300] + "...")


=== APERÇU DE VOTRE BASE DE DONNÉES (DATAFRAME) ===

--- Les 5 premières lignes du tableau ---


,review,cleaned_review,sentiment
0,I don't know how or why this film has a meager...,know film meager rating imdb film accompanied ...,1
1,For a long time it seemed like all the good Ca...,long time seemed like good canadian actors hea...,1
2,Terry Gilliam's and David Peoples' teamed up t...,terry gilliam david peoples teamed create one ...,1
3,What is there to say about an anti-establishme...,say anti establishment film produced time colo...,1
4,This movie was made only 48 years after the en...,movie made years end civil war likely anticipa...,1



--- Comparaison d'un avis Positif et d'un avis Négatif ---

[AVIS POSITIF ORIGINAL] :
I don't know how or why this film has a meager rating on IMDb. This film, accompanied by "I am Curious: Blue" is a masterwork.<br /><br />The only thing that will let you down in this film is if you don't like the process of film, don't like psychology or if you were expecting hardcore pornographic ...

[AVIS POSITIF NETTOYÉ] :
know film meager rating imdb film accompanied curious blue masterwork thing let film like process film like psychology expecting hardcore pornographic ramming film want watch unwind film want see like masterpiece time attention care summaries may contain spoiler two main thing film blends whole film...


[AVIS NÉGATIF ORIGINAL] :
I was looking forward to seeing Bruce Willis in this, especially since I remember being mesmerised by the original when I was young.<br /><br />This movie is a perfect example of how movie companies can take a very good story and dumb it down until it